<a href="https://colab.research.google.com/github/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/colab/07_weighted_ensembles_and_downloads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Google Colab version of this notebook. A copy with all its output is [in the SOURSOP repository](https://github.com/holehouse-lab/soursop/blob/master/demo_examples/ped_examples/07_weighted_ensembles_and_downloads.ipynb).

# 7. Weighted ensembles, and working with files

Many PED ensembles are **weighted**: they come from reweighting a larger pool of conformers against experimental data, so the conformers are not equally likely. Ignoring the weights gives the wrong ensemble averages.

This notebook shows how to fetch and use PED's weights, and how to save ensembles and PED's data files to disk.

In [ ]:
# Colab does not come with SOURSOP, so install it first. This uses uv, which
# is much faster than pip (and is itself installed first if it is missing);
# --system installs into Colab's own Python rather than a virtual environment.
# The PED module is new in SOURSOP 2.0.8; if that version is not on PyPI yet,
# this installs the latest version from GitHub instead.
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q "soursop>=2.0.8" 2>/dev/null || uv pip install --system -q "git+https://github.com/holehouse-lab/soursop.git"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from soursop import ped

# Figure style: Arial throughout, text kept editable in Illustrator, no grid
# lines, and the xmgrace colour order (black, red, blue, green, purple).
# Colab does not have Arial, so there figures fall back to Liberation Sans
# (metrically identical to Arial) or matplotlib's default, DejaVu Sans
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.grid": False,
    "axes.prop_cycle": plt.cycler(color=["black", "red", "blue", "green", "purple"]),
})

# Downloaded ensembles are kept here and reused, so each is only fetched once
# per session (Colab deletes them when the session ends)
CACHE = "ped_cache"

from soursop.sstrajectory import SSTrajectory

## A weighted ensemble: Tau K18

`PED00192` is a Bayesian-weighted ensemble of the Tau K18 fragment. `ped.get_ensemble_weights()` returns one weight per conformer, normalised to sum to 1 (or `None` if PED has no weights for an ensemble).

In [ ]:
IDENTIFIER = "PED00192e001"
entry = ped.get_entry(IDENTIFIER)
print(entry)
print(entry.title)

w = ped.get_ensemble_weights(IDENTIFIER)
n_eff = 1.0 / np.sum(w**2)          # Kish effective sample size
print(f"\n{len(w)} conformers, weights from {w.min():.2g} to {w.max():.2g}")
print(f"Effective number of conformers: {n_eff:.1f} of {len(w)}")

The Kish effective sample size, 1 / Σ w², shows how many equally weighted conformers the ensemble is worth: here roughly half of them.

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 2.6))
order = np.argsort(w)[::-1]
ax.bar(np.arange(1, len(w) + 1), w[order], width=1.0, color="black")
ax.axhline(1 / len(w), color="red", lw=0.8, zorder=3)
ax.set_xlabel("Conformer (ranked by weight)")
ax.set_ylabel("Weight")
plt.tight_layout()
plt.show()

The red line is the uniform weight, 1/N.

## Using the weights

Every SOURSOP method that computes an ensemble average accepts `weights=`, and PED's normalised weights can be passed straight in.

In [ ]:
protein = ped.load_ensemble(IDENTIFIER, cache_dir=CACHE).proteinTrajectoryList[0]
print(f"{protein.n_frames} conformers, {protein.n_residues} residues")

rg = protein.get_radius_of_gyration()
re = protein.get_end_to_end_distance(mode="CA")
print(f"<Rg>: unweighted {rg.mean():.1f} A, weighted {protein.get_radius_of_gyration(weights=w):.1f} A")
print(f"<Re>: unweighted {re.mean():.1f} A, weighted {protein.get_end_to_end_distance(mode='CA', weights=w):.1f} A")

In [ ]:
bins = np.linspace(rg.min(), rg.max(), 16)
fig, ax = plt.subplots(figsize=(3.6, 2.7))
ax.hist(rg, bins=bins, density=True, histtype="step", color="black", label="unweighted")
ax.hist(rg, bins=bins, weights=w, density=True, histtype="step", color="red", label="weighted")
ax.set_xlabel("Radius of gyration (Å)")
ax.set_ylabel("Probability density (1/Å)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

The weights shift the ensemble towards more compact conformers. The distance map shows which residue pairs the reweighting moved closer together (blue) or further apart (red).

In [ ]:
unweighted, _ = protein.get_distance_map(mode="CA", verbose=False)
weighted, _ = protein.get_distance_map(mode="CA", weights=w, verbose=False)
diff = (weighted - unweighted) + (weighted - unweighted).T
limit = np.abs(diff).max()
n = protein.n_residues

fig, ax = plt.subplots(figsize=(3.9, 3.2))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-limit, vmax=limit, extent=[1, n, n, 1])
ax.set_xlabel("Residue index")
ax.set_ylabel("Residue index")
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("Weighted − unweighted distance (Å)")
plt.tight_layout()
plt.show()

## Larger reweighted ensembles

The same works for large ensembles. `PED07323` is a 988-conformer molecular dynamics ensemble of human IAPP, reweighted against experimental data (about 40 MB to download).

In [ ]:
w_iapp = ped.get_ensemble_weights("PED07323e001")
iapp = ped.load_ensemble("PED07323e001", cache_dir=CACHE).proteinTrajectoryList[0]
print(f"{iapp.n_frames} conformers, effective number {1 / np.sum(w_iapp**2):.0f}")
print(f"<Rg>: unweighted {iapp.get_radius_of_gyration().mean():.2f} A, "
      f"weighted {iapp.get_radius_of_gyration(weights=w_iapp):.2f} A")

PED also serves weights for ensembles whose conformers are equally likely, so an unweighted ensemble simply returns uniform weights:

In [ ]:
print(ped.get_ensemble_weights("PED00001e001"))

## Saving files

`ped.download_ensemble()` saves an ensemble as a multi-model PDB file (`<identifier>.pdb`) and returns its path. If the file is already there, it is reused rather than downloaded again. The file can be read back in with `SSTrajectory` like any other PDB file, or opened in PyMOL, VMD and so on.

In Colab, files are saved on the Colab machine rather than your computer. Download them from the file browser (the folder icon on the left), or with `google.colab.files.download(path)`.

In [ ]:
path = ped.download_ensemble("PED00001e001", directory=CACHE)
print(path)

traj = SSTrajectory(path, path)
print(traj)

`ped.download_all_data()` downloads everything PED holds for an ensemble in a single `.tar.gz`: the ensemble itself and its weights.

In [ ]:
import tarfile

archive = ped.download_all_data("PED00001e001", directory=CACHE)
print(archive)
with tarfile.open(archive) as tar:
    for member in tar.getmembers():
        print(f"  {member.name:20s} {member.size / 1e3:8.1f} kB")

## When things go wrong

Every failure raises `ped.PEDError`, a subclass of SOURSOP's `SSException`. Its `status` attribute holds the HTTP status code (`404` for something that does not exist), or `None` if PED could not be reached.

In [ ]:
try:
    ped.get_entry("PED99999")
except ped.PEDError as error:
    print("status:", error.status)
    print(error)